In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
transformation=pd.read_csv(' data_transformation_dataset_lab3.csv')

In [ ]:
transformation.info()

# Task 1: Feature Scaling - Standardization (Z-Score Normalization)

## Goal
Scale the **Income** column so that it has:
- **Mean = 0**
- **Standard Deviation = 1**

Standardization (also known as **Z-Score Normalization**) ensures that features with large numerical values do not dominate machine learning algorithms such as **Support Vector Machines (SVMs)**, **Neural Networks**, and **K-Nearest Neighbors (KNN)**.

---

## Instructions

### Step 1: Import the StandardScaler
Import `StandardScaler` from the `sklearn.preprocessing` module.

### Step 2: Standardize the Income Column
- Create an instance of `StandardScaler`.
- Fit the scaler on the **Income** column.
- Transform the values into standardized scores.

### Step 3: Create a New Column
Store the standardized values in a new column named:

`Income_Standardized`

### Step 4: Verify the Result
Print the following statistics for the new column:
- Mean (Expected: **approximately 0**)
- Standard Deviation (Expected: **approximately 1**)

---

## Expected Output

After standardization:

- **Mean ≈ 0**
- **Standard Deviation ≈ 1**

This confirms that the feature has been successfully standardized and is ready for use in machine learning models.

In [ ]:
scaler = StandardScaler()
transformation["Income_Standardized"] = scaler.fit_transform(transformation[["Income"]])

In [ ]:
print("Mean of standardized Income:", transformation["Income_Standardized"].mean())
print("Standard Deviation of standardized Income:", transformation["Income_Standardized"].std(ddof=0))

# Task 2: Feature Scaling - Min-Max Normalization

## Goal
Scale the **Credit_Score** column into a fixed range between **0 and 1** using **Min-Max Normalization**.

Min-Max Normalization rescales the values of a feature while preserving their relative differences. It is commonly used for algorithms such as **Neural Networks**, **K-Nearest Neighbors (KNN)**, and **Gradient Descent-based models**, where features should have the same scale.

---

## Instructions

### Step 1: Import the MinMaxScaler
Import `MinMaxScaler` from the `sklearn.preprocessing` module.

### Step 2: Normalize the Credit_Score Column
- Create an instance of `MinMaxScaler`.
- Fit the scaler on the **Credit_Score** column.
- Transform the values into the range **[0, 1]**.

### Step 3: Create a New Column
Store the normalized values in a new column named:

`Credit_Score_Normalized`

### Step 4: Verify the Result
Print the following statistics for the new column:
- Minimum value (Expected: **0.0**)
- Maximum value (Expected: **1.0**)

---

## Expected Output

After Min-Max Normalization:

- **Minimum Value = 0.0**
- **Maximum Value = 1.0**

This confirms that the **Credit_Score** feature has been successfully normalized and is ready for use in machine learning models.

In [ ]:
scaler = MinMaxScaler()
transformation["Credit_Score_Normalized"] = scaler.fit_transform(transformation[["Credit_Score"]])

In [ ]:
print("Minimum value:", transformation["Credit_Score_Normalized"].min())
print("Maximum value:", transformation["Credit_Score_Normalized"].max())

# Task 3: One-Hot Encoding (Nominal Data)

## Goal
Convert the **Region** column into machine-readable binary (0s and 1s) features using **One-Hot Encoding**.

One-Hot Encoding is used for **nominal categorical variables**, where the categories have **no natural order**. Each unique category is converted into a separate binary column, allowing machine learning models to interpret categorical data correctly.

---

## Instructions

### Step 1: Apply One-Hot Encoding
Use `pandas.get_dummies()` on the **Region** column to create binary indicator columns.

### Step 2: Avoid the Dummy Variable Trap
Set `drop_first=True` to remove the first category. This prevents **multicollinearity**, where one dummy variable can be perfectly predicted from the others.

### Step 3: Concatenate the Encoded Columns
Append the newly created binary columns to the original dataframe.

### Step 4: Verify the Result
Display the first few rows of the dataframe or print the column names to confirm that the encoded columns have been added successfully.

---

## Expected Output

The original **Region** column will be represented by multiple binary columns, for example:

| Region_North | Region_South | Region_West |
|--------------|--------------|-------------|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |
| 1 | 0 | 0 |

> **Note:** Only **n − 1** dummy columns are created because `drop_first=True` removes one category to avoid the dummy variable trap.

In [ ]:
region_dummies = pd.get_dummies(transformation["Region"], prefix="Region", drop_first=True)
transformation = pd.concat([transformation, region_dummies], axis=1)

In [ ]:
print(transformation.head())

In [ ]:
print("\nNewly created columns:",region_dummies.columns.tolist())

In [ ]:
encoder = OneHotEncoder(drop="first", sparse_output=False)
region_encoded = encoder.fit_transform(transformation[["Region"]])
region_encoded_transformation = pd.DataFrame(region_encoded,columns=encoder.get_feature_names_out(["Region"]),index=transformation.index)
transformation = pd.concat([transformation, region_encoded_transformation], axis=1)

In [ ]:
print(transformation.head())

In [ ]:
print("\nNew encoded columns:",region_encoded_transformation.columns.tolist())

# Task 4: Label / Ordinal Encoding (Ordinal Data)

## Goal
Transform the **Education** column into numerical values while preserving its natural ranking/order.

Since **Education** is an ordinal categorical variable, the categories have a meaningful relationship:

```
High School < Bachelors < Masters < PhD
```

The encoding should preserve this order.

---

## Encoding Mapping

| Education | Encoded Value |
|-----------|---------------|
| High School | 0 |
| Bachelors | 1 |
| Masters | 2 |
| PhD | 3 |

---

## Method 1: Using Pandas `.map()`

### Steps:
1. Create a mapping dictionary with the correct order.
2. Apply `.map()` on the Education column.
3. Store the encoded values in a new column:

`Education_Encoded`

---

## Method 2: Using Sklearn `OrdinalEncoder`

### Steps:
1. Import `OrdinalEncoder` from `sklearn.preprocessing`.
2. Define the category order manually.
3. Fit and transform the Education column.
4. Store the encoded values in:

`Education_Encoded`

---

## Expected Output

The original categorical values:

```
High School
Bachelors
Masters
PhD
```

will become:

```
0
1
2
3
```

while maintaining the correct ordinal relationship.

In [ ]:
education_mapping = {"High School": 0,"Bachelors": 1,"Masters": 2,"PhD": 3}
transformation["Education_Encoded"] = transformation["Education"].map(education_mapping)
print(transformation[["Education", "Education_Encoded"]].head())

In [ ]:
education_order = [["High School", "Bachelors", "Masters", "PhD"]]
encoder = OrdinalEncoder(categories=education_order)
transformation["Education_Encoded"] = encoder.fit_transform(transformation[["Education"]])
transformation["Education_Encoded"] = transformation["Education_Encoded"].astype(int)
print(transformation[["Education", "Education_Encoded"]].head())

# Task 5: Discretization / Binning

## Goal
Convert the continuous **Age** variable into categorical age groups (bins) using **discretization**.

Binning helps machine learning models capture **non-linear relationships** by grouping similar age ranges together instead of treating age as a continuous variable.

---

## Age Bins

The custom bin edges are:

```
[0, 25, 45, 65, 100]
```

The corresponding labels are:

| Age Range | Category |
|-----------|----------|
| 0 - 25 | Youth |
| 25 - 45 | Adult |
| 45 - 65 | Middle-Aged |
| 65 - 100 | Senior |

---

## Instructions

### Step 1: Define Bin Edges
Create a list containing the age boundaries:

`[0, 25, 45, 65, 100]`

### Step 2: Define Bin Labels
Create labels for each age group:

`['Youth', 'Adult', 'Middle-Aged', 'Senior']`

### Step 3: Apply pandas.cut()
Use `pandas.cut()` to categorize the **Age** column into the defined groups.

### Step 4: Store the Result
Save the categorized values in a new column:

`Age_Group`

### Step 5: Check Distribution
Use `value_counts()` to display the number of records in each age group.

---

## Expected Output

Example:

```
Age_Group
Adult          250
Middle-Aged    180
Youth          120
Senior          50
```

This confirms that the continuous **Age** feature has been successfully converted into categorical groups.

In [ ]:
bins = [0, 25, 45, 65, 100]
labels = ["Youth", "Adult", "Middle-Aged", "Senior"]
transformation["Age_Group"] = pd.cut(transformation["Age"],bins=bins,labels=labels,include_lowest=True)
print(transformation["Age_Group"].value_counts())

# Task 6: Log Transformation (Handling Data Skewness)

## Goal
Reduce the heavy **right-skewness** in the `Transaction_Amount` column by applying a **log transformation**.

Many real-world financial datasets contain transaction amounts where a small number of very large transactions create a long right tail. Log transformation compresses these large values and makes the distribution closer to a normal (Gaussian) distribution.

---

## Instructions

### Step 1: Visualize Original Distribution
Plot a histogram of the original `Transaction_Amount` column using **Matplotlib** or **Seaborn**.

Observe:
- Long right tail
- Presence of highly skewed values

---

### Step 2: Apply Log Transformation
Use NumPy's:

```python
np.log1p()
```

`log1p(x)` calculates:

```
log(1 + x)
```

It is safer than `log(x)` because it can handle zero values without causing mathematical errors.

---

### Step 3: Create New Column
Store the transformed values in a new column:

```
Transaction_Amount_Log
```

---

### Step 4: Visualize Transformed Distribution
Plot a histogram of the new `Transaction_Amount_Log` column.

The transformed distribution should show:
- Reduced skewness
- Shorter tail
- More normalized shape

---

## Expected Result

Before transformation:

```
Transaction_Amount
        |
        |       *
        |       *
        |      **
        |____************
                 -------->
                 Long Tail
```

After log transformation:

```
Transaction_Amount_Log
        |
        |      ****
        |    ********
        |  **********
        |************
                 -------->
          More balanced distribution
```

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(transformation["Transaction_Amount"], bins=50, kde=True)
plt.title("Original Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.show()

In [ ]:
transformation["Transaction_Amount_Log"] = np.log1p(transformation["Transaction_Amount"])
plt.figure(figsize=(8, 5))
sns.histplot(transformation["Transaction_Amount_Log"], bins=50, kde=True)
plt.title("Log Transformed Transaction Amount Distribution")
plt.xlabel("Log(Transaction Amount + 1)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print(transformation[["Transaction_Amount", "Transaction_Amount_Log"]].head())